# 5.1 Comparaison des modèles (Axe A)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

RUNS_DIR     = Path('/root/Projet_Image/runs')
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'

RESULTS = {
    'YOLOv8m':      {'mAP50': 41.5,  'inf_ms': 244.8, 'gpu_gb': 6.4, 'train_min': 6.9},
    'Faster R-CNN': {'mAP50': 43.1,  'inf_ms': 91.3,  'gpu_gb': 6.5, 'train_min': 64.8},
    'SSD300':        {'mAP50': 19.4,  'inf_ms': 17.9,  'gpu_gb': 2.5, 'train_min': 48.9},
    'DETR':          {'mAP50': 0.0,   'inf_ms': 50.6,  'gpu_gb': 3.3, 'train_min': 86.6},
}

## 1. Tableau comparatif

In [ ]:
df = pd.DataFrame(RESULTS).T.rename(columns={
    'mAP50':     'mAP50 (%)',
    'inf_ms':    'Inférence (ms/img)',
    'gpu_gb':    'GPU (GB)',
    'train_min': 'Entraînement (min)',
})
df = df.sort_values('mAP50 (%)', ascending=False)
print(df.to_string())

## 2. Graphiques comparatifs

In [ ]:
modeles   = list(df.index)
couleurs  = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Comparaison des 4 modèles — 1 epoch, split val', fontsize=13, fontweight='bold')

# mAP50
axes[0].bar(modeles, df['mAP50 (%)'], color=couleurs)
axes[0].set_title('mAP50 (%)')
axes[0].set_ylabel('mAP50 (%)')
axes[0].set_ylim(0, 55)
for i, v in enumerate(df['mAP50 (%)']):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

# Inférence
axes[1].bar(modeles, df['Inférence (ms/img)'], color=couleurs)
axes[1].set_title('Temps d\'inférence (ms/img)')
axes[1].set_ylabel('ms / image')
for i, v in enumerate(df['Inférence (ms/img)']):
    axes[1].text(i, v + 2, f'{v:.1f}', ha='center', fontsize=10)

# GPU
axes[2].bar(modeles, df['GPU (GB)'], color=couleurs)
axes[2].set_title('Mémoire GPU (GB)')
axes[2].set_ylabel('GB')
axes[2].set_ylim(0, 8)
for i, v in enumerate(df['GPU (GB)']):
    axes[2].text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=10)

for ax in axes:
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('/root/Projet_Image/comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé : /root/Projet_Image/comparaison_modeles.png')

## 3. Conclusion

| Critère | Gagnant |
|---|---|
| Meilleure précision (mAP50) | Faster R-CNN (43.1%) |
| Inférence la plus rapide | SSD300 (17.9 ms) |
| Moins de VRAM | SSD300 (2.5 GB) |
| Entraînement le plus rapide | YOLOv8m (6.9 min) |

**Modèle retenu : YOLOv8m**

Faster R-CNN devance YOLOv8m de 1.6 points de mAP50 à 1 epoch, mais YOLOv8m est retenu pour deux raisons principales :
1. **Vitesse d'entraînement** : 9× plus rapide (6.9 vs 64.8 min), ce qui permet d'itérer sur les hyperparamètres et la résolution d'entrée ;
2. **Performance à convergence** : entraîné 100 epochs (voir doc 04), YOLOv8m atteint mAP50=0.651 — l'écart à 1 epoch n'est pas représentatif de la performance finale.

DETR (mAP50=0.0) confirme qu'une architecture à transformeurs nécessite bien davantage d'epochs pour converger — ce résultat est attendu et documenté (Carion et al., 2020).

## 4. Taille des modèles (nombre de paramètres, poids en Mo)

La consigne de l'axe A demande la taille de chaque modèle (nombre de paramètres et poids du fichier en Mo). Le code ci-dessous (non exécuté dans cette session) instancie chaque architecture et calcule ces deux valeurs.

- Pour **YOLOv8m**, le fichier de poids réellement entraîné existe (`BEST_WEIGHTS = runs/yolov8m_epi_1024-2/weights/best.pt`, voir notebook `2_4`) → taille en Mo mesurable directement.
- Pour **Faster R-CNN / SSD300 / DETR**, seul le nombre de paramètres de l'architecture (têtes adaptées à 17 classes) est calculé ici ; la taille en Mo du poids dépend du checkpoint sauvegardé lors du run à 1 epoch (section 1) — à compléter si ces fichiers sont conservés sur le serveur.

In [ ]:
# Code de mesure de la taille des modèles (non exécuté dans cette session — voir section 4)
from pathlib import Path
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, ssd300_vgg16
from transformers import DetrForObjectDetection


def n_parametres(model):
    return sum(p.numel() for p in model.parameters())


def taille_fichier_mo(chemin):
    return Path(chemin).stat().st_size / (1024 ** 2)


TAILLES = {}

# YOLOv8m — checkpoint réellement entraîné, taille du fichier mesurable directement
yolo = YOLO(str(BEST_WEIGHTS))
TAILLES['YOLOv8m'] = {
    'parametres_M': n_parametres(yolo.model) / 1e6,
    'poids_Mo':      taille_fichier_mo(BEST_WEIGHTS),
}

# Faster R-CNN / SSD300 / DETR — architectures non finetunées ici : nombre de paramètres
# de l'architecture de base (têtes adaptées à 17 classes), poids non mesurés (pas de
# checkpoint .pt sauvegardé pour ces 1-epoch runs dans cette session)
frcnn = fasterrcnn_resnet50_fpn_v2(weights=None, num_classes=18)
TAILLES['Faster R-CNN'] = {'parametres_M': n_parametres(frcnn) / 1e6, 'poids_Mo': None}

ssd = ssd300_vgg16(weights=None, num_classes=18)
TAILLES['SSD300'] = {'parametres_M': n_parametres(ssd) / 1e6, 'poids_Mo': None}

detr = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50', num_labels=17, ignore_mismatched_sizes=True
)
TAILLES['DETR'] = {'parametres_M': n_parametres(detr) / 1e6, 'poids_Mo': None}

for nom, infos in TAILLES.items():
    poids = f"{infos['poids_Mo']:.1f} Mo" if infos['poids_Mo'] else 'non mesuré'
    print(f"{nom:15s} : {infos['parametres_M']:6.1f} M paramètres  —  {poids}")

## 5. Temps d'inférence CPU vs GPU

Le temps d'inférence du tableau (section 1, colonne `inf_ms`) a été mesuré **sur GPU uniquement**. La consigne demande une mesure séparée CPU et GPU. Le code ci-dessous (non exécuté dans cette session, plus d'entraînement/calcul lancé — fournit la méthode reproductible) mesure le temps moyen par image sur `cpu` et `cuda` via `time.perf_counter`, avec un warmup pour éviter de mesurer le coût de chargement initial du modèle.

À titre indicatif, l'ordre de grandeur attendu (architectures de cette taille, image 1024×1024) est un facteur ×10 à ×30 entre CPU et GPU — c'est ce facteur, plus que la valeur absolue, qui doit guider le choix d'un modèle pour un déploiement temps réel sans GPU.

In [ ]:
# Code de mesure du temps d'inférence CPU vs GPU (non exécuté dans cette session — voir section 5)
# Adapter MODELES_A_MESURER avec les checkpoints réellement disponibles sur le serveur.
import time
import torch

def mesurer_temps_inference(predict_fn, image, device, n_repeats=20, n_warmup=3):
    """Mesure le temps moyen d'inférence (ms/image) sur 'cpu' ou 'cuda'."""
    for _ in range(n_warmup):
        predict_fn(image, device)
    if device == 'cuda':
        torch.cuda.synchronize()

    debut = time.perf_counter()
    for _ in range(n_repeats):
        predict_fn(image, device)
    if device == 'cuda':
        torch.cuda.synchronize()
    duree = time.perf_counter() - debut

    return duree / n_repeats * 1000  # ms / image


# Exemple d'utilisation avec YOLOv8m (à adapter pour Faster R-CNN / SSD300 / DETR) :
#
# from ultralytics import YOLO
# image = cv2.imread(str(TEST_IMAGES_DIR / chemins[0]))
# yolo = YOLO(str(BEST_WEIGHTS))
#
# for device in ['cpu', 'cuda']:
#     t = mesurer_temps_inference(
#         lambda img, dev: yolo.predict(img, device=dev, verbose=False),
#         image, device,
#     )
#     print(f'YOLOv8m  — {device:4s} : {t:.1f} ms/image')

## 6. Limites de la comparaison à 1 epoch (mAP50-95 et split test)

Les résultats du tableau (section 1) sont mesurés **à 1 epoch sur le split val**, dans une optique de tri rapide entre architectures avant de choisir celle à entraîner à convergence. Le **mAP50-95** n'est volontairement pas reporté à ce stade : à 1 epoch, les boîtes prédites sont encore mal calibrées et le mAP50-95 (plus exigeant sur l'IoU) serait proche de 0 pour les 4 modèles — peu informatif pour départager les architectures.

L'évaluation rigoureuse (mAP50, mAP50-95, précision, rappel, F1 par classe) **sur le split test**, à convergence (100 epochs), n'est disponible que pour le modèle retenu (YOLOv8m, 1024×1024) — voir `documentation/04.Evaluation.md` §2.4 : mAP50 = 0.697, mAP50-95 = 0.412. Reproduire ce même protocole pour Faster R-CNN / SSD300 / DETR demanderait un entraînement complet de chaque architecture (plusieurs heures chacune, cf. section 1), ce qui sort du cadre de ce tri initial.

## 7. Comportement sur les cas difficiles (modèle retenu : YOLOv8m 1024×1024)

La comparaison à 1 epoch (sections 1-3) ne permet pas une analyse fine du comportement sur des cas difficiles — à ce stade d'entraînement, les modèles n'ont pas encore appris à détecter la plupart des classes (DETR : mAP50 = 0). L'analyse qualitative ci-dessous porte donc sur le **modèle finalement retenu** (YOLOv8m, 1024×1024, entraîné 100 epochs — voir doc04), en s'appuyant sur des cas déjà identifiés lors de l'inférence (doc05).

**1. Petits objets vs grands objets**
Le passage de 640 à 1024 améliore le rappel des petits objets comme `helmet` (doc04 §2.4), conformément à la justification de pré-traitement (doc01 §1.3 : 52.3 % des bounding boxes < 1 % de l'image). En revanche, `safety-vest` (objet de grande taille) ne bénéficie pas de cette augmentation de résolution — son écart Rappel−Précision se dégrade même (−0.130 → −0.218). La difficulté sur cette classe n'est donc pas liée à la résolution, mais probablement aux poses/occlusions variées des gilets.

**2. Classe rare en effondrement : `medical-suit`**
Avec très peu d'instances dans le dataset, `medical-suit` passe à F1 = 0 / Rappel = 0 à 1024×1024 (doc04 §2.4) — illustration directe du ratio de déséquilibre identifié en exploration (doc01 §1.2, ratio ~118x).

**3. Faux positifs "casque manquant" malgré un casque visible**
Sur la vidéo de tracking (doc05, « Limite persistante »), un casque bleu clairement visible (logo "...nk Safety") déclenche deux fois l'alerte "casque manquant" — voir `../rapport_images/video_02_limite_1.png` et `../rapport_images/video_03_limite_2.png`. Cas difficile typique : confiance de détection du casque instable, ou IoU casque/head insuffisant pour valider la règle de conformité.

**4. Sensibilité au seuil de confiance (`conf_min`)**
Sur `photo_chantier2.png` (doc05, axe E), `conf_min=0.80` produit deux fausses alertes ("gilet manquant", "gants manquants" — `../photo_chantier_non_conforme_080.png`), corrigées avec `conf_min=0.85` (`../photo_chantier_non_conforme_085.png`). Un même cas peut donc basculer entre conforme/non-conforme selon le réglage du seuil — point critique pour un système d'alerte en conditions réelles.